## 关于模型调用invoke的使用

### 1. invoke的传参
### 1.1 文本输入
举例:

In [14]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

LLM_MODEL = os.getenv("LLM_MODE2")
LLM_BASE_URL = os.getenv("LLM_BASE_URL2")
LLM_API_KEY = os.getenv("LLM_API_KEY2")

model = init_chat_model(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    model_provider="openai",
)

res = model.invoke("请用中文回答：你是谁？")

print(res)

content='你好！我是DeepSeek，由深度求索公司创造的AI助手。我可以帮你解答问题、提供信息或进行对话。有什么我可以帮你的吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 242, 'prompt_tokens': 90, 'total_tokens': 332, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 208, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 90}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': None, 'id': '4de4cefb-3f77-468b-bff8-d07965740b04', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f92a5-9e52-7b53-9dc6-ff2a62db2a30-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 90, 'output_tokens': 242, 'total_tokens': 332, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 208}}


### 1.2 字典列表
举例1:

In [3]:
message = [
    {"role":"system","content":"你是一个数学家"},
    {"role":"user","content":"介绍一下斐波那契数列"},
    {"role":"user","content":"你的身份是什么？"}
]
res = model.invoke(message)
print(res)

content="我是 MiniMax-M3,由 MiniMax 开发。我的身份是一个人工智能助手,可以帮助解答各种问题,包括数学问题。\n\n## 斐波那契数列简介\n\n**定义**\n\n斐波那契数列(英语:Fibonacci sequence)是一个经典的整数数列,定义如下:\n\n- **F(0) = 0**\n- **F(1) = 1**\n- **F(n) = F(n−1) + F(n−2)** (n ≥ 2)\n\n即从第 3 项开始,每一项都等于前两项之和。\n\n**前几项**\n\n| n | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |\n|---|---|---|---|---|---|---|---|---|---|---|----|\n| F(n) | 0 | 1 | 1 | 2 | 3 | 5 | 8 | 13 | 21 | 34 | 55 |\n\n**来源**\n\n该数列最早由意大利数学家**列奥纳多·斐波那契(Leonardo Fibonacci)**在 1202 年的著作《算盘书》(Liber Abaci)中提出。书中以**兔子繁殖问题**作为引入:\n\n> 假设一对刚出生的兔子,第二个月成熟,之后每个月产下一对新兔子,且兔子不会死亡。问 n 个月后有多少对兔子?\n\n答案正好就是斐波那契数列。\n\n**重要性质**\n\n1. **黄金比例联系**:相邻两项之比 F(n+1)/F(n) 趋近于**黄金比例 φ ≈ 1.618**。\n2. **黄金比例恒等式**:φ = (1 + √5)/2,且满足 φ² = φ + 1。\n3. **比内公式**(Binet's Formula,1843):\n   $$F(n) = \\frac{\\varphi^n - \\psi^n}{\\sqrt{5}}$$\n   其中 ψ = (1 − √5)/2 ≈ −0.618。\n4. **递推关系**:F(n+k) = F(k)·F(n+1) + F(k−1)·F(n)。\n5. **求和公式**:\n   $$\\sum_{i=0}^{n} F(i) = F(n+2) - 1$$\n\n**应用领域**\n\n斐波那契数列在数学、自然科学与艺术中均有广泛身影:\n\n- **数学**

举例2:没有记忆

In [4]:
message = [
    {"role":"system","content":"你是一个好用的ai助手"},
    {"role":"user","content":"你好，我是小明"}
]
res = model.invoke(message)
print("第一次对话",res.content)

message = [
    {"role":"user","content":"我是谁？"}
]
res = model.invoke(message)
print("第二次对话",res.content)

第一次对话 你好，小明！很高兴认识你！有什么我可以帮你的吗？
第二次对话 你是南昌（用户名为 **Nanchang**）😊


举例2:携带记忆

In [16]:
conversation = [
    {"role":"system","content":"你是一个好用的ai助手"},
    {"role":"user","content":"你好，我是小明"}
]

res = model.invoke(conversation)
print("第一次对话",res.content)

# 使用列表追加方法
conversation.append({"role":"assistant","content":res.content})
conversation.append({"role":"user","content":"我是谁？"})
res = model.invoke(conversation)
print("第二次对话",res.content)

第一次对话 你好呀，小明！很高兴认识你～有什么我可以帮你的吗？无论是学习、生活还是日常闲聊，尽管问我哦！😊
第二次对话 哈哈，你刚刚自己说过啦——你是小明！😊 如果这是个脑筋急转弯的话，那我也会坚定地站在“小明”这个答案上～ 有什么需要帮忙的吗？


### 1.3 消息对象列表
举例:

In [15]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

message = [
    SystemMessage(content="你是一个好用的ai助手"),
    HumanMessage(content="你好，我是小明")
]

res = model.invoke(message)
print("第一次对话",res.content)

message.append(AIMessage(content=res.content))
message.append(HumanMessage(content="我是谁？"))

res = model.invoke(message)
print("第二次对话",res.content)

第一次对话 content='你好，小明！很高兴认识你。我是DeepSeek，由深度求索公司创造的AI助手。有什么我可以帮你的吗？无论是聊天、解答问题还是帮忙处理信息，尽管开口！😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 93, 'total_tokens': 213, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 76, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 93}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': None, 'id': '9eb2d01c-c37d-4457-897c-1fa97b82de65', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f92a5-c44d-7972-b14f-5c1f2eb63a9d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 93, 'output_tokens': 120, 'total_tokens': 213, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 76}}
第二次对话 content='哈哈，你刚才不是说了吗——你是小明呀！😊\n\n你第一句就是“你好，我是小明”，所

### 美化输出
举例:

In [17]:
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage

message = [
    SystemMessage(content="你是一个好用的ai助手"),
    HumanMessage(content="你好，我是小明")
]

res = model.invoke(message)
rprint(res)

AIMessage(
    content='你好，小明！很高兴认识你！😊 我是你的AI助手，有什么想聊的或者需要帮忙的吗？',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 72,
            'prompt_tokens': 93,
            'total_tokens': 165,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 46,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 93
        },
        'model_provider': 'openai',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': None,
        'id': 'ed3f7bb5-ddc8-48a9-bd95-b641aff30ac2',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f92af-b577-72c1-9c32-0ba7b98622da-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 93,
        'output_tokens': 72,
        'total_tokens': 165,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 46}
    }
)